# Change from V2.0:
## Updated Secondary Label detection and allication in labeling stage

In [2]:
# %% [markdown]
# # RQ2 — Step 0: Clone repositories
# Reads a CSV with column `repo_url`, clones/fetches into CLONE_ROOT,
# and writes a manifest with basic metadata.

# %%
from __future__ import annotations
import csv, subprocess, sys, json, time
from pathlib import Path
from typing import Optional, List
from pathlib import Path

# -----------------------------
# Config (edit as needed)
# -----------------------------


# Use a raw string r"..." for Windows paths with spaces
WORK_ROOT    = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
URL_LIST_CSV = WORK_ROOT / "URL_List.csv"   # put your CSV here
CLONE_ROOT   = WORK_ROOT / "clones"         # repos will clone here
MANIFEST_CSV = WORK_ROOT / "clones_manifest.csv"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)


# Create dirs
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)

# %%
def sh(cmd: List[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, cwd=cwd, check=check, capture_output=True, text=True)

def repo_dir_name_from_url(url: str) -> str:
    # e.g. https://github.com/owner/name(.git) -> owner__name
    base = url.split("//")[-1]
    parts = base.split("/")
    if len(parts) >= 3:
        owner = parts[-2]
        name  = parts[-1].replace(".git", "")
        return f"{owner}__{name}"
    return base.replace("/", "__").replace(".git", "")

def ensure_cloned(url: str, dest_root: Path) -> Path:
    dest_root.mkdir(parents=True, exist_ok=True)
    d = dest_root / repo_dir_name_from_url(url)
    if d.exists() and (d / ".git").exists():
        # Refresh remote info (best-effort)
        try:
            sh(["git", "fetch", "--all", "--tags", "--prune"], cwd=d)
        except Exception:
            pass
        return d
    sh(["git", "clone", "--no-tags", "--filter=blob:none", "--recurse-submodules=no", url, str(d)])
    return d

def get_total_commits(repo_dir: Path) -> int:
    cp = sh(["git", "rev-list", "--all", "--count"], cwd=repo_dir)
    return int(cp.stdout.strip() or "0")

# %%
assert URL_LIST_CSV.exists(), f"CSV not found: {URL_LIST_CSV}"

rows, ok, fail = [], 0, 0
with URL_LIST_CSV.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        url = (row.get("repo_url") or "").strip()
        if not url:
            continue
        t0 = time.time()
        rec = {"repo_url": url, "dir": None, "status": "unknown", "seconds": None, "total_commits": None, "error": ""}
        try:
            d = ensure_cloned(url, CLONE_ROOT)
            rec["dir"] = str(d)
            rec["total_commits"] = get_total_commits(d)
            rec["status"] = "ok"
            ok += 1
        except subprocess.CalledProcessError as e:
            rec["status"] = "error"
            rec["error"]  = (e.stderr or e.stdout or str(e)).strip()[:2000]
            fail += 1
        rec["seconds"] = round(time.time() - t0, 2)
        rows.append(rec)
        print(f"[{rec['status']}] {url} -> {rec['dir']} ({rec['seconds']}s)")

# %%
# Write manifest
MANIFEST_CSV.parent.mkdir(parents=True, exist_ok=True)
with MANIFEST_CSV.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ["repo_url","dir","status","seconds","total_commits","error"])
    w.writeheader()
    w.writerows(rows)

print(f"\nDone. OK={ok}, FAIL={fail}. Manifest: {MANIFEST_CSV}")


[ok] https://github.com/connectbot/connectbot -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\connectbot__connectbot (1.64s)
[ok] https://github.com/robolectric/robolectric -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\robolectric__robolectric (7.22s)
[ok] https://github.com/opendocument-app/OpenDocument.droid -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\opendocument-app__OpenDocument.droid (4.29s)
[ok] https://github.com/maxpower47/PinDroid -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\maxpower47__PinDroid (2.95s)
[ok] https://github.com/Rajawali/Rajawali -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\Rajawali__Rajawali (6.46s)
[ok] https://github.com/cgeo/cgeo -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\cgeo__cgeo (24.03s)
[ok] https://github.com/OneBusAway/onebusaway-android -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\OneBusAway__onebus

In [2]:
# %% [markdown]
# RQ2 — Step 1: Mine commit snapshots (inclusive CI patterns + cutoff)
# Scans cloned repos for commits touching CI/YAML/Gradle/scripts and writes per-repo JSONL snapshots.
# - Windows-safe UTF-8 decoding for all git calls (prevents UnicodeDecodeError).
# - Robust fallbacks so a wonky repo doesn't stop the whole run.
# - Inclusive CI vendor path detection (Travis/AppVeyor/CircleCI/GHA/GitLab/Jenkins/etc.).
# - Reduced script false positives (extensions only).
# - Invocation STYLE detection (GMD / DIY / gradle_connected / unknown) — NO tags.
# - Richer AGP & Orchestrator detection.
# - Commit date cutoff: include commits up to end of Aug 10, 2025 (America/Toronto).

from __future__ import annotations

import os
import re
import json
import subprocess
import datetime as dt
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any

# ---- Optional tz support (Py 3.9+) ----
try:
    from zoneinfo import ZoneInfo  # Python 3.9+
except Exception:
    ZoneInfo = None

# -----------------------------
# Config (edit as needed)
# -----------------------------
WORK_ROOT      = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
CLONE_ROOT     = WORK_ROOT / "clones"
SNAPSHOT_DIR   = WORK_ROOT / "snapshots"   # per-repo JSONL output here
MAX_COMMITS_PER_REPO = 0                   # 0 = no limit

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Commit cutoff (America/Toronto) ----
CUTOFF_TZ_NAME = "America/Toronto"
_CUTOFF_DATE   = (2025, 8, 10, 23, 59, 59)  # YYYY, M, D, H, M, S local to Toronto
if ZoneInfo is not None:
    _tz = ZoneInfo(CUTOFF_TZ_NAME)
    _local_dt = dt.datetime(*_CUTOFF_DATE, tzinfo=_tz)
    CUTOFF_EPOCH = int(_local_dt.timestamp())
    CUTOFF_BEFORE_STR = _local_dt.strftime("%Y-%m-%d %H:%M:%S %z")
else:
    _utc_dt = dt.datetime(2025, 8, 11, 3, 59, 59, tzinfo=dt.timezone.utc)  # EDT fallback
    CUTOFF_EPOCH = int(_utc_dt.timestamp())
    CUTOFF_BEFORE_STR = "2025-08-10 23:59:59 -0400"

print(f"[cutoff] Using commit cutoff <= {CUTOFF_BEFORE_STR} (epoch={CUTOFF_EPOCH})")

# Optional YAML support (safe to skip if you don't need deep YAML parsing)
try:
    import yaml  # pip install pyyaml
except Exception:
    yaml = None

# -----------------------------
# Subprocess helper (Windows-safe UTF-8)
# -----------------------------
def sh(cmd: List[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    env = os.environ.copy()
    env["GIT_PAGER"] = ""
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        check=check,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",
        errors="replace",
        env=env,
    )

# -----------------------------
# Relevant file surfaces (INCLUSIVE CI VENDOR LIST)
# -----------------------------
CI_VENDOR_PATTERNS = [
    re.compile(r'(?i)(?:^|/)\.travis\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.appveyor\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)appveyor\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)circle\.yml$'),
    re.compile(r'(?i)(?:^|/)\.circleci/config\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)azure-pipelines\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.github/workflows/.*\.(yml|yaml)$'),
    re.compile(r'(?i)(?:^|/)bitbucket-pipelines\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.gitlab-ci\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)Jenkinsfile(?:\.\w+)?$'),
    re.compile(r'(?i)(?:^|/)bitrise\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)bamboo\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)codeship-services\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.gocd\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.cirrus\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)wercker\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)semaphore\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)codemagic\.ya?ml$'),
]
GENERIC_CI_DIRS = re.compile(r'(?i)(?:^|/)(?:ci|\.ci|\.jenkins)(?:/|$)')

def is_ci_file(p: str) -> bool:
    if not p:
        return False
    for rx in CI_VENDOR_PATTERNS:
        if rx.search(p):
            return True
    return bool(GENERIC_CI_DIRS.search(p))

GRADLE_FILES = [
    "build.gradle", "build.gradle.kts",
    "settings.gradle", "settings.gradle.kts",
    "gradle.properties",
    "gradle/wrapper/gradle-wrapper.properties",
]
SCRIPT_EXTS = {".sh", ".bash", ".zsh", ".py", ".bat", ".cmd", ".ps1", ".psm1"}

def is_gradle_file(p: str) -> bool:
    lp = p.lower()
    return any(lp.endswith(x) for x in (f.lower() for f in GRADLE_FILES))

def is_script_file(p: str) -> bool:
    lp = p.lower()
    return any(lp.endswith(ext) for ext in SCRIPT_EXTS)

def touched_relevant(paths: List[str]) -> bool:
    for p in paths:
        if not p.strip():
            continue
        if is_ci_file(p) or is_gradle_file(p) or is_script_file(p):
            return True
    return False

# -----------------------------
# Git helpers
# -----------------------------
def list_relevant_commits(repo_dir: Path) -> List[Tuple[str, int, List[str]]]:
    cp = sh([
        "git", "-c", "i18n.logOutputEncoding=UTF-8", "-c", "core.quotepath=off",
        "log", "--all",
        "--before", CUTOFF_BEFORE_STR,
        "--name-only", "--pretty=%H%x09%ct"
    ], cwd=repo_dir)
    results: List[Tuple[str, int, List[str]]] = []
    sha: Optional[str] = None
    ts: Optional[int] = None
    changed: List[str] = []
    for line in cp.stdout.splitlines():
        if re.match(r"^[0-9a-f]{40}\t\d+$", line):
            if sha is not None and ts is not None and ts <= CUTOFF_EPOCH and touched_relevant(changed):
                results.append((sha, ts, changed))
            sha, ts_s = line.split("\t", 1)
            ts = int(ts_s)
            changed = []
        else:
            if line.strip():
                changed.append(line.strip())
    if sha is not None and ts is not None and ts <= CUTOFF_EPOCH and touched_relevant(changed):
        results.append((sha, ts, changed))
    results.reverse()
    return results

def git_show(repo_dir: Path, sha: str, path: str) -> Optional[str]:
    try:
        cp = sh(["git", "show", f"{sha}:{path}"], cwd=repo_dir)
        return cp.stdout
    except subprocess.CalledProcessError:
        return None

def git_subject(repo_dir: Path, sha: str) -> str:
    try:
        cp = sh(["git", "-c", "i18n.logOutputEncoding=UTF-8", "show", "-s", "--format=%s", sha],
                cwd=repo_dir, check=False)
        return (cp.stdout or "").strip()
    except Exception:
        return ""

# -----------------------------
# Heuristic extractors
# -----------------------------
RE_INT = re.compile(r"\d+")

YAML_KEYS = {
    "api": ["api-level","apilevel","api_level"],
    "abi": ["abi","arch","cpu","abi_filters","abi-filter"],
    "system_image": ["system-image","target","systemimage"],
    "device": ["device","avd-name","avd","device-profile","model","hardwareProfile"],
    "orchestrator": ["orchestrator","android-test-orchestrator","use-orchestrator"],
    "wait": ["wait-for-boot","wait_for_boot"],
    "timeouts": ["emulator-boot-timeout","timeout","test-timeout","emulator_timeout"],
    "retries": ["retry","retries","max-retries"],
    "matrix": ["matrix","strategy"],
    "runner_os": ["runs-on","machine","image"],
    "jdk": ["java-version","jdk","java","distribution"],
    "invocation": ["run","gradle_args","gradlew_args","task","tasks"],
    # NOTE: thirdparty providers removed from mining
}

# --- Invocation style classification (NO tags) ---
DIY_RX = re.compile(
    r"(?:\bemulator(?:\.bat)?\s-|"
    r"\bavdmanager\b|"
    r"\bsdkmanager\b|"
    r"\bcreate\s+avd\b|"
    r"\badb\s+(?:-s\s+\S+\s+)?wait-for-device\b|"
    r"\badb\s+shell\s+getprop\s+sys\.boot_completed\b|"
    r"\bqemu\b)",
    re.I,
)

GMD_RX = re.compile(
    r"(?:\bmanaged\s+device\b|\bgradle\s+managed\s+device\b|\bPixel\w*Api\d+\b)",
    re.I,
)

GMD_GRADLE_RX = re.compile(
    r"(?:\bmanagedDevices\s*\{|testOptions\s*\{[^}]*devices)",
    re.I | re.S,
)

CONNECTED_RX = re.compile(r"\bconnectedAndroidTest\b", re.I)

def classify_invocation_style(yaml_snippets: List[str], gradle_texts: List[str]) -> str:
    """
    Returns invocation_style only (gmd / diy / gradle_connected / unknown)
    """
    hay_yaml = "\n".join(s for s in yaml_snippets if s)
    hay_gradle = "\n".join(gradle_texts or [])

    gmd_hit = bool(GMD_RX.search(hay_yaml)) or any(GMD_GRADLE_RX.search(t or "") for t in gradle_texts or [])
    diy_hit = bool(DIY_RX.search(hay_yaml) or DIY_RX.search(hay_gradle))
    connected_hit = bool(CONNECTED_RX.search(hay_yaml) or CONNECTED_RX.search(hay_gradle))

    if gmd_hit:
        return "gmd"
    if diy_hit:
        return "diy"
    if connected_hit:
        return "gradle_connected"
    return "unknown"

# -----------------------------
# YAML extractor
# -----------------------------
def extract_from_yaml_text(text: str, gradle_texts: Optional[List[str]] = None) -> Dict[str, Any]:
    if yaml is None:
        return {}
    try:
        docs = list(yaml.safe_load_all(text))
    except Exception:
        docs = []
    out = {
        # real features
        "api_levels": set(), "abis": set(), "system_images": set(), "device_profiles": set(),
        "orchestrator": None, "wait_for_boot": None, "timeouts": {}, "retries": None,
        "matrix_axes": set(), "runner_os": None, "jdk": None,
        # study-defined (simplified)
        "invocation_style": "unknown",
        # NOTE: thirdparty_refs removed
    }
    _invocation_snippets: List[str] = []

    def scan_obj(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                lk = str(k).lower()
                if lk in (x.lower() for x in YAML_KEYS["api"]):
                    if isinstance(v, list):
                        for x in v:
                            if isinstance(x, (int, str)) and RE_INT.search(str(x)):
                                out["api_levels"].add(int(RE_INT.search(str(x)).group()))
                    elif isinstance(v, (int, str)):
                        m = RE_INT.search(str(v))
                        if m:
                            out["api_levels"].add(int(m.group()))
                if lk in (x.lower() for x in YAML_KEYS["abi"]):
                    vals = v if isinstance(v, list) else [v]
                    for x in vals:
                        if isinstance(x, str):
                            out["abis"].add(x.strip())
                if lk in (x.lower() for x in YAML_KEYS["system_image"]):
                    vals = v if isinstance(v, list) else [v]
                    for x in vals:
                        if isinstance(x, str):
                            out["system_images"].add(x.strip())
                if lk in (x.lower() for x in YAML_KEYS["device"]):
                    vals = v if isinstance(v, list) else [v]
                    for x in vals:
                        if isinstance(x, str):
                            out["device_profiles"].add(x.strip())
                if lk in (x.lower() for x in YAML_KEYS["orchestrator"]):
                    if isinstance(v, bool):
                        out["orchestrator"] = v
                    elif isinstance(v, str):
                        out["orchestrator"] = v.lower() in ("1", "true", "yes", "on")
                if lk in (x.lower() for x in YAML_KEYS["wait"]):
                    if isinstance(v, bool):
                        out["wait_for_boot"] = v
                    elif isinstance(v, str):
                        out["wait_for_boot"] = v.lower() in ("1", "true", "yes", "on")
                if lk in (x.lower() for x in YAML_KEYS["timeouts"]):
                    out["timeouts"][k] = v
                if lk in (x.lower() for x in YAML_KEYS["retries"]):
                    try:
                        out["retries"] = int(RE_INT.search(str(v)).group())
                    except Exception:
                        pass
                if lk in (x.lower() for x in YAML_KEYS["matrix"]):
                    if isinstance(v, dict):
                        for ax, _vals in v.items():
                            out["matrix_axes"].add(str(ax))
                if lk in (x.lower() for x in YAML_KEYS["runner_os"]):
                    out["runner_os"] = str(v)
                if lk in (x.lower() for x in YAML_KEYS["jdk"]):
                    out["jdk"] = str(v)
                if lk in (x.lower() for x in YAML_KEYS["invocation"]):
                    _invocation_snippets.append(str(v))
                if isinstance(v, (dict, list)):
                    scan_obj(v)
        elif isinstance(obj, list):
            for x in obj:
                scan_obj(x)

    for d in docs:
        scan_obj(d)

    # classify invocation using YAML snippets + Gradle context (if provided)
    out["invocation_style"] = classify_invocation_style(_invocation_snippets, gradle_texts or [])

    # convert sets to sorted lists for JSON
    out["api_levels"]      = sorted(out["api_levels"])
    out["abis"]            = sorted(out["abis"])
    out["system_images"]   = sorted(out["system_images"])
    out["device_profiles"] = sorted(out["device_profiles"])
    out["matrix_axes"]     = sorted(out["matrix_axes"])
    return out

# -----------------------------
# Gradle helpers (richer detection)
# -----------------------------
AGP_PLUGIN_DSL_RX = re.compile(
    r"""id\s*\(?\s*      # id(
        [\"']com\.android\.(?:application|library|test|dynamic-feature)[\"']\s*\)?   # id("com.android.xyz")
        \s*version\s*
        [\"']([^\"']+)[\"']                      # version "X.Y.Z"
    """,
    re.I | re.X,
)

ORCHESTRATOR_COORD_RX = re.compile(r"androidx\.test:orchestrator(?::[^\s'\"\)]+)?", re.I)
ORCHESTRATOR_EXEC_RX  = re.compile(r"testOptions\s*\{[^}]*execution\s*['\"]ANDROIDX_TEST_ORCHESTRATOR['\"]", re.I | re.S)
ORCHESTRATOR_FLAG_RX  = re.compile(r"\buseOrchestrator\s*(?:=|\s)\s*true\b", re.I)
ORCHESTRATOR_PROP_RX  = re.compile(r"\bandroid(?:\.testInstrumentationRunnerArguments)?\.use(?:Test)?Orchestrator\s*=\s*true", re.I)

def detect_orchestrator_from_gradle(text: str) -> bool:
    t = text or ""
    return bool(
        ORCHESTRATOR_COORD_RX.search(t) or
        ORCHESTRATOR_EXEC_RX.search(t)  or
        ORCHESTRATOR_FLAG_RX.search(t)  or
        ORCHESTRATOR_PROP_RX.search(t)
    )

# -----------------------------
# Single-file extractor (YAML + Gradle)
# -----------------------------
def extract_from_text(path: str, text: str, gradle_context_texts: Optional[List[str]] = None) -> Dict[str, Any]:
    data: Dict[str, Any] = {}

    # YAML configs (with gradle context for richer classification)
    if path.lower().endswith((".yml", ".yaml")) and yaml is not None:
        data = extract_from_yaml_text(text, gradle_context_texts or [])

    # Gradle heuristics
    if path.endswith(("build.gradle", "build.gradle.kts",
                      "gradle.properties", "gradle/wrapper/gradle-wrapper.properties",
                      "settings.gradle", "settings.gradle.kts")):
        # AGP via dependency coordinates (classpath or anywhere)
        dep_agp = re.findall(r"com\.android\.tools\.build:gradle:([0-9][^'\"\s\)]+)", text)
        # AGP via plugins { id("com.android.application") version "X" }
        dsl_agp = AGP_PLUGIN_DSL_RX.findall(text)
        agp_all = sorted(set(dep_agp + dsl_agp))
        if agp_all:
            data["agp_versions"] = agp_all

        # apiLevel = N (managed devices DSL or custom config)
        for m in re.finditer(r"\bapiLevel\s*=\s*(\d+)", text, re.IGNORECASE):
            lvl = int(m.group(1))
            data.setdefault("api_levels", [])
            if lvl not in data["api_levels"]:
                data["api_levels"].append(lvl)

        # Orchestrator signals
        if "ANDROIDX_TEST_ORCHESTRATOR" in text or detect_orchestrator_from_gradle(text):
            data["orchestrator"] = True

        # Recognize GMD via Gradle DSL (prefer gmd if present)
        if GMD_GRADLE_RX.search(text):
            current = data.get("invocation_style")
            if current in (None, "", "unknown", "diy", "gradle_connected"):
                data["invocation_style"] = "gmd"

        # Gradle wrapper version (optional)
        if path.endswith("gradle/wrapper/gradle-wrapper.properties"):
            m = re.search(r"distributionUrl=.*?/gradle-([0-9][\w\.\-]+)-", text)
            if m:
                data["gradle_wrapper_version_raw"] = m.group(1)

    return data

# =============================
# NEW: Labeling helpers
# =============================

# --- Diff added-line numbers (for change-location labeling) ---
DIFF_HUNK_RX = re.compile(r"@@ -\d+(?:,\d+)? \+(\d+)(?:,(\d+))? @@")

def added_line_numbers_for_path(repo_dir: Path, sha: str, path: str) -> Optional[set[int]]:
    """
    Returns the set of 1-based line numbers added in `path` at `sha`.
    None if diff can't be read.
    """
    try:
        cp = sh(["git", "-c", "core.quotepath=off", "diff", "-U0", f"{sha}^", sha, "--", path],
                cwd=repo_dir, check=False)
    except Exception:
        return None
    if cp.returncode not in (0, 1):  # 1 = diff found
        return None

    added: set[int] = set()
    new_line = None
    for line in cp.stdout.splitlines():
        m = DIFF_HUNK_RX.match(line)
        if m:
            start = int(m.group(1))
            length = int(m.group(2) or "1")
            new_line = start
            continue
        if new_line is None:
            continue
        if line.startswith("+") and not line.startswith("+++"):
            added.add(new_line)
            new_line += 1
        elif line.startswith("-") and not line.startswith("---"):
            # deletion doesn't advance new file line counter
            pass
        else:
            if not (line.startswith("---") or line.startswith("+++")):
                new_line += 1
    return added

# --- File kind ---
def file_kind_for_path(p: str) -> str:
    if is_gradle_file(p):
        return "gradle"
    if is_ci_file(p) and p.lower().endswith((".yml", ".yaml")):
        return "ci_yaml"
    if is_script_file(p):
        return "script"
    return "other"

# --- CI YAML: emulator step spans (indent + run/script blocks w/ DIY_RX) ---
STEP_START_RX = re.compile(r"""^(\s*)-\s+(?:name:\s*.*)?\s*$""", re.I | re.M)
RUN_KEY_RX    = re.compile(r"""^(\s*)(run|script)\s*:\s*(\|>|)?\s*$""", re.I | re.M)
LIST_ITEM_RX  = re.compile(r"""^\s*-\s+""")

def find_emulator_step_line_spans(yaml_text: str) -> list[tuple[int,int]]:
    lines = yaml_text.splitlines()
    n = len(lines)
    spans: list[tuple[int,int]] = []
    i = 0
    while i < n:
        mstep = STEP_START_RX.match(lines[i])
        if not mstep:
            i += 1
            continue
        step_indent = len(mstep.group(1) or "")
        j = i + 1
        while j < n:
            line = lines[j]
            if STEP_START_RX.match(line):
                break
            cur_indent = len(line) - len(line.lstrip(" "))
            if cur_indent < step_indent and line.strip():
                break
            mrun = RUN_KEY_RX.match(line)
            if mrun:
                block_indent = len(mrun.group(1) or "")
                k = j + 1
                block_idx: list[int] = []
                while k < n:
                    ln = lines[k]
                    if not ln.strip():
                        block_idx.append(k); k += 1; continue
                    ind = len(ln) - len(ln.lstrip(" "))
                    if ind <= block_indent and not LIST_ITEM_RX.match(ln):
                        break
                    block_idx.append(k); k += 1
                block_text = "\n".join(lines[x] for x in block_idx)
                if DIY_RX.search(block_text):
                    if block_idx:
                        spans.append((min(block_idx)+1, max(block_idx)+1))
                j = k
                continue
            j += 1
        i = j
    # merge
    merged: list[list[int]] = []
    for s,e in sorted(spans):
        if not merged or s > merged[-1][1] + 1:
            merged.append([s,e])
        else:
            merged[-1][1] = max(merged[-1][1], e)
    return [(s,e) for s,e in merged]

# --- Gradle: GMD block spans via brace scan ---
GMD_BLOCK_HEAD_RX = re.compile(r"""
    (?P<head>
        (?:\bmanagedDevices\s*\{)|
        (?:\btestOptions\s*\{\s*(?:[^{}]*\{[^{}]*\}[^{}]*)*\s*devices\s*\{)
    )
""", re.I | re.X | re.S)

def _find_block_span_from_head(text: str, head_start: int) -> tuple[int,int] | None:
    i = text.find("{", head_start)
    if i == -1:
        return None
    depth = 0
    for j in range(i, len(text)):
        ch = text[j]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return (i, j)
    return None

def _char_span_to_line_span(text: str, ci: tuple[int,int]) -> tuple[int,int]:
    start_char, end_char = ci
    start_line = text[:start_char].count("\n") + 1
    end_line   = text[:end_char+1].count("\n") + 1
    return (start_line, end_line)

def find_gmd_line_spans(gradle_text: str) -> list[tuple[int,int]]:
    spans: list[tuple[int,int]] = []
    for m in GMD_BLOCK_HEAD_RX.finditer(gradle_text):
        ci = _find_block_span_from_head(gradle_text, m.start())
        if ci:
            spans.append(_char_span_to_line_span(gradle_text, ci))
    return spans

# --- Overlap helpers ---
def lines_overlap_spans(added: set[int] | None, spans: list[tuple[int,int]]) -> Optional[bool]:
    if added is None:
        return None
    if not spans:
        return False
    for a in added:
        for s,e in spans:
            if s <= a <= e:
                return True
    return False

# --- Alias-driven locators for feature-scope labeling ---
ALIASES = {
    "api_levels":    ["api-level", "apilevel", "api_level"],
    "abis":          ["abi", "arch", "cpu", "abi_filters", "abi-filter"],
    "system_images": ["system-image", "target", "systemimage"],
    "device_profiles":["device", "avd-name", "avd", "device-profile", "model", "hardwareProfile"],
    "orchestrator":  ["orchestrator", "android-test-orchestrator", "use-orchestrator"],
    "wait_for_boot": ["wait-for-boot", "wait_for_boot"],
    "timeouts":      ["emulator-boot-timeout", "timeout", "test-timeout", "emulator_timeout"],
    "retries":       ["retry", "retries", "max-retries"],
    "matrix_axes":   ["matrix", "strategy"],
    "runner_os":     ["runs-on", "machine", "image"],
    "jdk":           ["java-version", "jdk", "java", "distribution"],
    "invocation_style": ["run", "gradle_args", "gradlew_args", "task", "tasks"],
}

def _alias_token_rx(tok: str) -> str:
    esc = re.escape(tok)
    return rf"(?:^|\s|^-)\s*{esc}\s*(?::|=|\s|$)"

def build_feature_locators(aliases: dict[str, list[str]]) -> dict[str, re.Pattern]:
    locators: dict[str, re.Pattern] = {}
    for feat, toks in aliases.items():
        parts = [_alias_token_rx(t) for t in toks]
        locators[feat] = re.compile("|".join(parts), re.I | re.M)
    return locators

FEATURE_LOCATORS = build_feature_locators(ALIASES)

ALWAYS_EMULATOR_FIELDS = {
    "api_levels", "abis", "system_images", "device_profiles",
    "invocation_style", "orchestrator",
}
ALWAYS_CONTEXT_FIELDS = {"runner_os", "jdk", "matrix_axes"}
CONDITIONAL_EMULATOR_FIELDS = {"timeouts", "wait_for_boot", "retries"}

def _line_hits(text: str, rx: re.Pattern) -> set[int]:
    hits: set[int] = set()
    for m in rx.finditer(text):
        hits.add(text.count("\n", 0, m.start()) + 1)
    return hits

def _any_line_in_spans(lines: set[int], spans: list[tuple[int,int]]) -> bool:
    if not lines or not spans:
        return False
    for ln in lines:
        for s, e in spans:
            if s <= ln <= e:
                return True
    return False

def compute_feature_scope(
    path: str,
    text: str,
    feats: dict,
    file_kind: str,
    ci_emulator_spans: list[tuple[int,int]] | None,
    gmd_spans: list[tuple[int,int]] | None,
) -> dict:
    """
    Returns {"field": "emulator_related" | "context_only"} for each key in `feats`.
    Rules:
      - ALWAYS_EMULATOR_FIELDS => emulator_related
      - ALWAYS_CONTEXT_FIELDS  => context_only
      - CONDITIONAL_EMULATOR_FIELDS => emulator_related iff alias hit is inside emulator step/GMD span
      - Everything else => context_only by default
    """
    scope: dict[str, str] = {}
    spans = ci_emulator_spans if file_kind == "ci_yaml" else (gmd_spans if file_kind == "gradle" else [])

    # Always emulator-related
    for f in ALWAYS_EMULATOR_FIELDS:
        if f in feats:
            scope[f] = "emulator_related"

    # Always context-only
    for f in ALWAYS_CONTEXT_FIELDS:
        if f in feats:
            scope[f] = "context_only"

    # Conditional fields
    for f in CONDITIONAL_EMULATOR_FIELDS:
        if f not in feats:
            continue
        rx = FEATURE_LOCATORS.get(f)
        in_block = False
        if rx:
            hits = _line_hits(text, rx)
            in_block = _any_line_in_spans(hits, spans or [])
        scope[f] = "emulator_related" if in_block else "context_only"

    # Default the rest to context_only unless already labeled
    for f in feats.keys():
        if f not in scope:
            scope[f] = "context_only"

    return scope

# -----------------------------
# IO helpers
# -----------------------------
def write_jsonl(path: Path, rows: List[dict]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# -----------------------------
# Main mining loop
# -----------------------------
def main():
    repos = [p for p in CLONE_ROOT.iterdir() if (p / ".git").exists()]
    repos.sort(key=lambda p: p.name.lower())
    print(f"Found {len(repos)} repos in {CLONE_ROOT}")

    ok_count = 0
    skip_count = 0
    err_count = 0

    for repo in repos:
        try:
            rel_commits = list_relevant_commits(repo)
            if MAX_COMMITS_PER_REPO > 0:
                rel_commits = rel_commits[:MAX_COMMITS_PER_REPO]

            out_rows: List[dict] = []
            for sha, ts, changed_paths in rel_commits:
                rel_paths = [p for p in changed_paths if is_ci_file(p) or is_gradle_file(p) or is_script_file(p)]
                if not rel_paths:
                    continue

                path_texts: Dict[str, Optional[str]] = {}
                for pth in rel_paths:
                    path_texts[pth] = git_show(repo, sha, pth)

                gradle_texts = [txt for pth, txt in path_texts.items() if txt is not None and is_gradle_file(pth)]

                subj = git_subject(repo, sha)
                for pth in rel_paths:
                    txt = path_texts.get(pth)
                    if txt is None:
                        continue

                    feats = extract_from_text(pth, txt, gradle_context_texts=gradle_texts)

                    # --- NEW: labels ---
                    fk = file_kind_for_path(pth)
                    added_lines = added_line_numbers_for_path(repo, sha, pth)

                    # Spans for this file
                    ci_spans  = find_emulator_step_line_spans(txt) if fk == "ci_yaml" else None
                    gmd_spans = find_gmd_line_spans(txt)           if fk == "gradle"  else None

                    ci_emu_change  = lines_overlap_spans(added_lines, ci_spans or []) if fk == "ci_yaml" else None
                    gmd_block_change = lines_overlap_spans(added_lines, gmd_spans or []) if fk == "gradle" else None

                    labels = {
                        "file_kind": fk,
                        "ci_emulator_step_change": ci_emu_change,
                        "gradle_gmd_block_change": gmd_block_change,
                    }

                    # Feature scope (emulator_related vs context_only)
                    feature_scope = compute_feature_scope(
                        path=pth,
                        text=txt,
                        feats=feats,
                        file_kind=fk,
                        ci_emulator_spans=ci_spans,
                        gmd_spans=gmd_spans,
                    )

                    # Store labels inside features (and you can also top-level if you want)
                    feats["labels"] = labels
                    feats["feature_scope"] = feature_scope

                    out_rows.append({
                        "repo": repo.name,
                        "sha": sha,
                        "timestamp": ts,
                        "subject": subj,
                        "path": pth,
                        "features": feats,
                    })

            if not out_rows:
                print(f"[skip] {repo.name}: no relevant snapshots")
                skip_count += 1
                continue

            dst = SNAPSHOT_DIR / f"{repo.name}.jsonl"
            write_jsonl(dst, out_rows)
            print(f"[ok] {repo.name}: {len(out_rows)} snapshots -> {dst}")
            ok_count += 1

        except Exception as e:
            print(f"[err] {repo.name}: {e}")
            err_count += 1

    print(f"\nDone. ok={ok_count}, skip={skip_count}, err={err_count}, out_dir={SNAPSHOT_DIR}")

if __name__ == "__main__":
    main()


[cutoff] Using commit cutoff <= 2025-08-10 23:59:59 -0400 (epoch=1754884799)
Found 282 repos in C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones
[ok] 4eRTuk__audioview: 78 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\4eRTuk__audioview.jsonl
[ok] a-mabe__OpenHIIT: 171 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\a-mabe__OpenHIIT.jsonl
[ok] a914-gowtham__compose-ratingbar: 159 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\a914-gowtham__compose-ratingbar.jsonl
[ok] AAkira__ExpandableLayout: 56 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\AAkira__ExpandableLayout.jsonl
[ok] abdelaziz-mahdy__pytorch_lite: 131 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\abdelaziz-mahdy__pytorch_lite.jsonl
[ok] ably__ably-flutter: 316 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\ably__abl

In [4]:
# RQ2 — Step 2, V4.7 (Delta trimmed & per-CCE; Subject=4 intents; PathFA=STRICT)
# - Per-source intents: intent_delta, intent_subject, intent_path, intent_pfa
# - Arbitration (Strategy A): Delta → Path-FA → Subject → Path
# - Driver = single 7-bucket intent; intent_headline = collapse to 4 buckets
# - Secondary_Label = comma-joined intents from the chosen source ONLY
# - Evidence fields: Driver_Source, Driver_reason
# - Categories aligned to tables A–E: Coverage, Invocation, Orchestration, Runtime, Context
# - Step-1 labels/scope carried over:
#     labels.file_kind, labels.ci_emulator_step_change, labels.gradle_gmd_block_change,
#     feature_scope[field] (emulator_related|context_only)
# - New derived fields:
#     Spot_Tag (yaml_scoped|yaml_unscoped|gradle_gmd|gradle_global|None)
#     Intent_Label (emulator_intent|context_only)
#     Old_Field_Scope / New_Field_Scope / Effective_Field_Scope
#     Include_Emulator_Study (bool)
# - Explicitly EXCLUDES:
#     invocation_tag (never produced)
#     matrix_axes / matrix_change (dropped completely)

from __future__ import annotations
import json, re
import datetime as _dt
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional

# -----------------------------
# Config
# -----------------------------
WORK_ROOT         = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
SNAPSHOT_DIR      = WORK_ROOT / "snapshots"
OUT_DIR           = WORK_ROOT / "cce_enriched_V4.7"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Small helpers
# -----------------------------
VERSION_RE = re.compile(r"\d+(?:\.\d+)*")

def read_snapshots(folder: Path) -> Dict[str, List[dict]]:
    by_repo: Dict[str, List[dict]] = {}
    for p in folder.glob("*.jsonl"):
        with p.open(encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                d = json.loads(line)
                repo = d.get("repo")
                if not repo:
                    continue
                by_repo.setdefault(repo, []).append(d)
    for repo, rows in by_repo.items():
        rows.sort(key=lambda r: (r.get("path",""), int(r.get("timestamp", 0)), r.get("sha","")))
    return by_repo

def as_set_str(xs) -> set:
    if xs is None: return set()
    if isinstance(xs, (list, set, tuple)):
        return set(str(x) for x in xs)
    return {str(xs)}

def as_set_int(xs) -> set:
    if xs is None: return set()
    out = set()
    if isinstance(xs, (list, set, tuple)):
        for x in xs:
            try: out.add(int(x))
            except: pass
    else:
        try: out.add(int(xs))
        except: pass
    return out

def stringify(x: Any) -> str:
    if isinstance(x, (dict, list, set, tuple)):
        try: return json.dumps(x, ensure_ascii=False, sort_keys=True)
        except: return str(x)
    return "" if x is None else str(x)

def _safe_json_loads(s: str):
    try: return json.loads(s) if s else None
    except: return None

def parse_version_tuple(s: str) -> Tuple[int, ...]:
    m = VERSION_RE.search(str(s) if s is not None else "")
    if not m: return tuple()
    parts = m.group(0).split(".")
    out: List[int] = []
    for p in parts:
        try: out.append(int(p))
        except: out.append(0)
    return tuple(out)

def max_version_tuple(strings: List[str]) -> Tuple[int, ...]:
    best: Tuple[int, ...] = tuple()
    for s in strings or []:
        vt = parse_version_tuple(str(s))
        if vt > best: best = vt
    return best

# -----------------------------
# Primary label mapping (WHAT) → Category (tables A–E)
# -----------------------------
FIELD_TO_PRIMARY = {
    "api_levels":        "api_bump",
    "agp_versions":      "agp_bump",
    "jdk":               "jdk_bump",
    "runner_os":         "runner_os_change",
    "orchestrator":      "orchestrator_change",
    "timeouts":          "timeout_tuning",
    "retries":           "retry_tuning",
    "device_profiles":   "device_profile_change",
    "abis":              "abi_change",
    "system_images":     "system_image_change",
    "invocation_style":  "invocation_change",
    "wait_for_boot":     "wait_strategy_change",
    # NOTE: invocation_tag removed; matrix_axes dropped
}

# Categories aligned to A–E tables
CAT_COVERAGE = "Coverage"        # A/D: what you boot
CAT_INVOC    = "Invocation"      # B: execution policy
CAT_ORCH     = "Orchestration"   # B: instrumentation infra toggle
CAT_RUNTIME  = "Runtime"         # C: scoped runtime tuning
CAT_CONTEXT  = "Context"         # E: toolchain/host/CI

CATEGORY_NAME = {
    CAT_COVERAGE: "Coverage",
    CAT_INVOC:    "Invocation",
    CAT_ORCH:     "Orchestration",
    CAT_RUNTIME:  "Runtime",
    CAT_CONTEXT:  "Context",
}

PRIMARY_TO_CATEGORY = {
    # A / D Coverage
    "api_bump":              CAT_COVERAGE,
    "abi_change":            CAT_COVERAGE,
    "device_profile_change": CAT_COVERAGE,
    "system_image_change":   CAT_COVERAGE,

    # B Invocation & Orchestration
    "invocation_change":     CAT_INVOC,
    "orchestrator_change":   CAT_ORCH,

    # C Runtime tuning
    "timeout_tuning":        CAT_RUNTIME,
    "retry_tuning":          CAT_RUNTIME,
    "wait_strategy_change":  CAT_RUNTIME,

    # E Context / Toolchain / Host / CI
    "runner_os_change":      CAT_CONTEXT,
    "jdk_bump":              CAT_CONTEXT,
    "agp_bump":              CAT_CONTEXT,

    # Fallback
    "other_change":          CAT_CONTEXT,
}

# -----------------------------
# Diff engine → per-field CCEs
# -----------------------------
def diff_features(old: Dict[str, Any], new: Dict[str, Any]) -> List[Dict[str, Any]]:
    old = old or {}; new = new or {}
    out: List[Dict[str, Any]] = []

    def handle_set(field: str, to_set_fn):
        a = to_set_fn(old.get(field)); b = to_set_fn(new.get(field))
        if a == b: return
        added   = sorted(b - a)
        removed = sorted(a - b)
        row = {
            "field": field,
            "old_value": stringify(sorted(a)),
            "new_value": stringify(sorted(b)),
            "change_type": "modified",
        }
        if added:   row["added_items"] = stringify(added)
        if removed: row["removed_items"] = stringify(removed)
        if field == "api_levels" and a and b:
            try: row["magnitude"] = max(b) - max(a)
            except: pass
        out.append(row)

    # Set-like (study + emulator fields)
    handle_set("api_levels", as_set_int)
    handle_set("abis", as_set_str)
    handle_set("system_images", as_set_str)
    handle_set("device_profiles", as_set_str)
    # matrix_axes dropped

    # AGP versions (ordered strings)
    a_agp = sorted(as_set_str(old.get("agp_versions"))); b_agp = sorted(as_set_str(new.get("agp_versions")))
    if a_agp != b_agp:
        row = {"field":"agp_versions","old_value":stringify(a_agp),"new_value":stringify(b_agp),"change_type":"modified"}
        old_max = max_version_tuple(list(a_agp)); new_max = max_version_tuple(list(b_agp))
        if old_max or new_max:
            row["magnitude"] = 1 if new_max > old_max else (-1 if new_max < old_max else 0)
        out.append(row)

    # JDK version (scalar string) — magnitude block
    a_jdk = stringify(old.get("jdk"))
    b_jdk = stringify(new.get("jdk"))
    if a_jdk != b_jdk:
        row = {
            "field": "jdk",
            "old_value": a_jdk,
            "new_value": b_jdk,
            "change_type": ("modified" if (a_jdk and b_jdk) else ("added" if not a_jdk else "removed")),
        }
        old_vt = parse_version_tuple(a_jdk)
        new_vt = parse_version_tuple(b_jdk)
        if old_vt or new_vt:
            row["magnitude"] = 1 if new_vt > old_vt else (-1 if new_vt < old_vt else 0)
        out.append(row)

    # Scalars (exclude jdk here—we handled it above)
    for field in ("orchestrator","wait_for_boot","retries","runner_os","invocation_style"):
        a = old.get(field, None); b = new.get(field, None)
        if a != b:
            out.append({
                "field": field, "old_value": stringify(a), "new_value": stringify(b),
                "change_type": ("modified" if (a is not None and b is not None)
                                else ("added" if a is None else "removed")),
            })

    # Dict-ish
    for field in ("timeouts",):
        a = old.get(field, None); b = new.get(field, None)
        if stringify(a) != stringify(b):
            out.append({
                "field": field, "old_value": stringify(a), "new_value": stringify(b),
                "change_type": ("modified" if (a is not None and b is not None)
                                else ("added" if a is None else "removed")),
            })
    return out

# -----------------------------
# Delta whitelist (ONLY these fields can emit Delta intents)
# -----------------------------
DELTA_MEANINGFUL_FIELDS = {
    # coverage (emulator sets ONLY)
    "api_levels", "abis", "device_profiles", "system_images",
    # runtime knobs
    "timeouts", "retries",
    # invocation flips
    "invocation_style",
    # versions
    "agp_versions", "jdk",
}

# -----------------------------
# Delta-side intents (meaningful-only)
# -----------------------------
_DURATION_RX = re.compile(r"(?i)^\s*(\d+(?:\.\d+)?)\s*(ms|s|m|h)?\s*$")
_UNIT_TO_SEC = {"ms": 0.001, "s": 1, "m": 60, "h": 3600}

def _to_seconds(x) -> Optional[float]:
    if x is None: return None
    if isinstance(x, (int, float)): return float(x)
    m = _DURATION_RX.match(str(x))
    if not m: return None
    val = float(m.group(1)); unit = (m.group(2) or "s").lower()
    return val * _UNIT_TO_SEC.get(unit, 1)

def _sum_timeout_seconds(obj) -> Optional[float]:
    if obj is None: return None
    total = 0.0; seen = 0
    if isinstance(obj, dict):
        it = obj.values()
    elif isinstance(obj, list):
        it = [v for _, v in obj if isinstance(_, (str,int))] if obj and isinstance(obj[0], (list,tuple)) else obj
    else:
        return None
    for v in it:
        secs = _to_seconds(v)
        if secs is not None:
            total += secs; seen += 1
    return total if seen else None

def derive_delta_intents(deltas: List[Dict[str, Any]]) -> List[str]:
    """
    Emit intents from Delta ONLY for whitelisted fields, and only where the delta adds intent-level meaning:
      - Coverage sets: expand_coverage / reduce_coverage (when both sides non-empty)
      - Timeouts: ↑ flake_mitigation, ↓ speed_up_ci
      - Retries:  ↑ flake_mitigation, ↓ speed_up_ci
      - Invocation: adopt_gmd / drop_gmd / adopt_diy
      - Versions (AGP/JDK): version_change when magnitude != 0
    """
    labs: set[str] = set()
    for d in deltas:
        field = d.get("field")
        if field not in DELTA_MEANINGFUL_FIELDS:
            continue

        old_v = d.get("old_value"); new_v = d.get("new_value")
        added = _safe_json_loads(d.get("added_items") or "") or []
        removed = _safe_json_loads(d.get("removed_items") or "") or []

        # Coverage sets
        if field in {"api_levels", "abis", "device_profiles", "system_images"}:
            old_list = _safe_json_loads(old_v) or []
            new_list = _safe_json_loads(new_v) or []
            if old_list and new_list:
                if added:   labs.add("expand_coverage")
                if removed: labs.add("reduce_coverage")

        # Timeouts
        if field == "timeouts":
            old_obj = _safe_json_loads(old_v); new_obj = _safe_json_loads(new_v)
            old_s = _sum_timeout_seconds(old_obj); new_s = _sum_timeout_seconds(new_obj)
            if old_s is not None and new_s is not None:
                if new_s > old_s: labs.add("flake_mitigation")
                elif new_s < old_s: labs.add("speed_up_ci")

        # Retries
        if field == "retries":
            def _to_int(x):
                if x is None or (isinstance(x, str) and not x.strip()): return None
                try: v = json.loads(x)
                except Exception: v = x
                try: return int(v)
                except Exception: return None
            a = _to_int(old_v); b = _to_int(new_v)
            if a is not None and b is not None:
                if b > a: labs.add("flake_mitigation")
                elif b < a: labs.add("speed_up_ci")

        # Invocation
        if field == "invocation_style":
            o = (old_v or "").strip().lower()
            n = (new_v or "").strip().lower()
            if n == "gmd" and o != "gmd": labs.add("adopt_gmd")
            if o == "gmd" and n != "gmd": labs.add("drop_gmd")
            if n == "diy" and o != "diy": labs.add("adopt_diy")

        # Versions
        if field in {"agp_versions","jdk"}:
            mag = d.get("magnitude", None)
            if mag is not None and mag != 0:
                labs.add("version_change")

    return sorted(labs)

# -----------------------------
# Path detectors (generic + STRICT field-aware)
# -----------------------------
CI_VENDOR_PATTERNS = [
    (re.compile(r'(?i)(?:^|/)\.travis\.ya?ml$'),               'Travis_CI'),
    (re.compile(r'(?i)(?:^|/)\.appveyor\.ya?ml$'),             'AppVeyor'),
    (re.compile(r'(?i)(?:^|/)/?appveyor\.ya?ml$'),             'AppVeyor'),
    (re.compile(r'(?i)(?:^|/)/?circle\.yml$'),                 'Circle_CI'),
    (re.compile(r'(?i)(?:^|/)\.circleci/config\.ya?ml$'),      'Circle_CI'),
    (re.compile(r'(?i)(?:^|/)/?azure-pipelines\.ya?ml$'),      'Azure_Pipelines'),
    (re.compile(r'(?i)(?:^|/)\.github/workflows/.*\.(yml|yaml)$'), 'GitHub_Actions'),
    (re.compile(r'(?i)(?:^|/)/?bitbucket-pipelines\.ya?ml$'),  'Bitbucket'),
    (re.compile(r'(?i)(?:^|/)\.gitlab-ci\.ya?ml$'),            'GitLab'),
    (re.compile(r'(?i)(?:^|/)/?Jenkinsfile(?:\.ya?ml)?$'),     'Jenkins'),
    (re.compile(r'(?i)(?:^|/)/?bitrise\.ya?ml$'),              'Bitrise'),
    (re.compile(r'(?i)(?:^|/)/?bamboo\.ya?ml$'),               'Bamboo'),
    (re.compile(r'(?i)(?:^|/)/?codeship-services\.ya?ml$'),    'Codeship'),
    (re.compile(r'(?i)(?:^|/)\.gocd\.ya?ml$'),                 'GoCD'),
    (re.compile(r'(?i)(?:^|/)\.cirrus\.ya?ml$'),               'Cirrus'),
    (re.compile(r'(?i)(?:^|/)/?wercker\.ya?ml$'),              'Wercker'),
    (re.compile(r'(?i)(?:^|/)\.semaphore\.ya?ml$'),            'Semaphore'),
    (re.compile(r'(?i)(?:^|/)/?codemagic\.ya?ml$'),            'Nevercode'),
]
GENERIC_CI_DIRS   = re.compile(r'(?i)(?:^|/)(?:ci|\.ci)(?:/|$)')
_GRADLE_PATH_RX   = re.compile(r"(?i)(?:^|/)(?:build|settings)\.gradle(?:\.kts)?$|(?:^|/)gradle\.properties$|(?:^|/)gradle/wrapper/gradle-wrapper\.properties$")
_SCRIPT_PATH_RX   = re.compile(r"(?i)\.(?:sh|py|bat|ps1)$|(?:^|/)(?:scripts?|tools?)(?:/|$)")

STRICT_PFA = True

# CI env fields (thirdparty removed)
CI_ENV_FIELDS = {"runner_os"}

def detect_intent_path_generic(path: str) -> List[str]:
    intents: List[str] = []
    p = path or ""
    if any(rx.search(p) for rx, _ in CI_VENDOR_PATTERNS) or GENERIC_CI_DIRS.search(p):
        intents.append("ci_env_workflow")
    if _GRADLE_PATH_RX.search(p):
        intents.append("version_change")
    if _SCRIPT_PATH_RX.search(p):
        intents.append("ci_env_workflow")
    return sorted(set(intents))

def detect_intent_path_field_aware(path: str,
                                   field: str,
                                   subject: str = "",
                                   delta_row: Optional[Dict[str, Any]] = None) -> Tuple[List[str], Optional[str]]:
    """
    STRICT PathFA:
      Emit only when BOTH a path pattern hits AND the field is relevant.

      Rules (matrix_axes removed):
        • CI vendor/CI dir path + {api_levels, abis, device_profiles, system_images}
            -> ["coverage", "ci_env_workflow"]
        • CI vendor/CI dir path + {runner_os} -> ["ci_env_workflow"]
        • Gradle/tooling path + {agp_versions, jdk} -> ["version_change"]
        • Script paths only reinforce ci_env_workflow for CI env fields.
    """
    p = path or ""; f = (field or "").lower()
    intents: List[str] = []
    reasons: List[str] = []

    vendor = None
    for rx, name in CI_VENDOR_PATTERNS:
        if rx.search(p):
            vendor = name; break
    if vendor is None and GENERIC_CI_DIRS.search(p):
        vendor = "Generic_CI"

    is_gradle_path = bool(_GRADLE_PATH_RX.search(p))
    is_script_path = bool(_SCRIPT_PATH_RX.search(p))

    if vendor:
        if f in {"api_levels", "abis", "device_profiles", "system_images"}:
            intents += ["coverage", "ci_env_workflow"]
            reasons.append(f"ci vendor={vendor} + coverage field")
        elif f in CI_ENV_FIELDS:
            intents += ["ci_env_workflow"]
            reasons.append(f"ci vendor={vendor} + {f}")

    if is_gradle_path and f in {"agp_versions", "jdk"}:
        intents.append("version_change")
        reasons.append("gradle/build tool path")

    if is_script_path and f in CI_ENV_FIELDS:
        intents.append("ci_env_workflow")
        reasons.append("script/tool path + ci field")

    intents = sorted(set(intents))
    reason = "; ".join(reasons) if reasons else None
    return intents, reason

# -----------------------------
# Subject-driven intents (4 driver-feeding; simplified keywords)
# -----------------------------
def detect_subject_intents(subject: str) -> Tuple[List[str], List[str], Optional[str]]:
    s = (subject or "").lower()

    hits = set()
    # flake_mitigation
    if any(w in s for w in ["flake", "flaky", "retry", "retries", "timeout", "timeouts", "stabil", "crash", "fix"]):
        hits.add("flake_mitigation")
    # speed_up_ci
    if any(w in s for w in ["speed up","faster","reduce time","time to green","parallel","shard","concurr"]) and \
       any(c in s for c in [" ci", "build", "pipeline", "workflow", "runner", "gha", "github actions", "gitlab", "jenkins", "circleci", "azure pipelines", "bitrise"]):
        hits.add("speed_up_ci")
    # version_change
    if any(w in s for w in ["gradle","agp","jdk","java","version","wrapper","target api"]):
        hits.add("version_change")
    # ci_env_workflow
    if any(w in s for w in ["migrate","switch","replace","port","github actions","gha","gitlab","jenkins","circleci","azure pipelines","bitrise","workflow","pipeline"]):
        hits.add("ci_env_workflow")

    hits = sorted(hits)
    reason = hits[0] if hits else None
    return hits, [], reason

# -----------------------------
# 7-subintent + 4-headline mapping
# -----------------------------
RAW_TO_SUBINTENT = {
    # coverage
    "expand_coverage":"coverage", "reduce_coverage":"coverage",
    "coverage_dimensions_change":"coverage", "coverage":"coverage",
    # runtime
    "flake_mitigation":"flake_mitigation", "timeout_tuning":"flake_mitigation",
    "retry_tuning":"flake_mitigation", "speed_up_ci":"speed_up_ci",
    "orchestrator_change":"orchestrator_change",
    # version/toolchain
    "version_change":"version_change", "adopt_gmd":"invocation_change",
    "drop_gmd":"invocation_change", "adopt_diy":"invocation_change",
    # ci
    "ci_env_workflow":"ci_platform_infra", "infra_tuning":"ci_platform_infra",
}
SUB_TO_HEADLINE = {
    "coverage":"coverage",
    "flake_mitigation":"runtime",
    "speed_up_ci":"runtime",
    "orchestrator_change":"runtime",
    "version_change":"version_change",
    "invocation_change":"version_change",
    "ci_platform_infra":"ci_platform_infra",
}
SUB_PRIORITY = ["coverage","flake_mitigation","speed_up_ci","version_change","orchestrator_change","invocation_change","ci_platform_infra"]

def to_subbuckets(raws: List[str]) -> List[str]:
    out = []
    for r in raws or []:
        b = RAW_TO_SUBINTENT.get(r)
        if b and b not in out:
            out.append(b)
    return out

def pick_by_priority(cands: List[str]) -> str:
    for b in SUB_PRIORITY:
        if b in cands:
            return b
    return ""

# -----------------------------
# Change-op utils
# -----------------------------
def classify_change_op(old_value: str, new_value: str, change_type: str) -> str:
    ct = (change_type or "").lower()
    if ct == "added":   return "add"
    if ct == "removed": return "remove"
    return "value_edit" if (old_value or "") != (new_value or "") else "no_change"

def ensure_utc_epoch(ts_any) -> int:
    try: ts = int(float(ts_any))
    except: ts = 0
    return ts

def epoch_to_iso_utc(ts: int) -> str:
    return _dt.datetime.fromtimestamp(int(ts), tz=_dt.timezone.utc).isoformat().replace("+00:00", "Z")

# -----------------------------
# Step-1 labels/scope helpers
# -----------------------------
def _get_labels(feats: dict) -> dict:
    return (feats or {}).get("labels", {}) or {}

def _get_scope(feats: dict) -> dict:
    return (feats or {}).get("feature_scope", {}) or {}

def _scope_keys(scope: dict, want: str) -> List[str]:
    return sorted([k for k, v in (scope or {}).items() if v == want])

def _is_emulator_relevant_file(labels: dict, scope: dict) -> bool:
    if any(v == "emulator_related" for v in (scope or {}).values()):
        return True
    if labels.get("ci_emulator_step_change") is True:
        return True
    if labels.get("gradle_gmd_block_change") is True:
        return True
    return False

# -----------------------------
# Spot tag & Intent label (tables A–E)
# -----------------------------
def _spot_tag(file_kind: str, field_scope: str) -> Optional[str]:
    if file_kind == "ci_yaml":
        return "yaml_scoped" if field_scope == "emulator_related" else "yaml_unscoped"
    if file_kind == "gradle":
        return "gradle_gmd" if field_scope == "emulator_related" else "gradle_global"
    return None

def _intent_label(primary_label: str,
                  primary_category: str,
                  file_kind: str,
                  field: str,
                  field_scope: str) -> str:
    # A: Coverage
    if primary_category == CAT_COVERAGE:
        return "emulator_intent"
    # B: Invocation & Orchestration
    if primary_category in (CAT_INVOC, CAT_ORCH):
        return "emulator_intent"
    # C: Runtime tuning — YAML scoped vs unscoped
    if primary_category == CAT_RUNTIME:
        if file_kind == "ci_yaml":
            return "emulator_intent" if field_scope == "emulator_related" else "context_only"
        return "context_only"
    # E: Context
    return "context_only"

# -----------------------------
# Effective scope for conditional runtime fields
# -----------------------------
CONDITIONAL_RUNTIME = {"timeouts","retries","wait_for_boot"}

def effective_scope_for_conditional(field: str, old_scope_val: str, new_scope_val: str) -> str:
    """
    For conditional runtime fields:
      - emulator_related if either side is emulator_related
      - context_only if both sides are context_only (or missing/empty)
    For all other fields, return the new scope (Step-1 scope remains authoritative).
    """
    f = (field or "").lower()
    o = (old_scope_val or "").strip().lower()
    n = (new_scope_val or "").strip().lower()
    if f in CONDITIONAL_RUNTIME:
        if o == "emulator_related" or n == "emulator_related":
            return "emulator_related"
        return "context_only"
    return n or ""

# -----------------------------
# Main
# -----------------------------
if __name__ == "__main__":
    print(f"[info] snapshots dir: {SNAPSHOT_DIR}")
    print(f"[info] output dir   : {OUT_DIR}")

    by_repo = read_snapshots(SNAPSHOT_DIR)
    print(f"[info] Loaded snapshots for {len(by_repo)} repos")

    for repo, rows in by_repo.items():
        out_rows: List[dict] = []
        by_path: Dict[str, List[dict]] = {}
        for r in rows:
            by_path.setdefault(r.get("path",""), []).append(r)

        repeat_counter: Dict[Tuple[str,str], int] = {}

        for path, snaps in by_path.items():
            prev: Optional[dict] = None
            for cur in snaps:
                if prev is None:
                    prev = cur
                    continue

                old_feats = prev.get("features", {}) or {}
                new_feats = cur.get("features", {}) or {}
                deltas = diff_features(old_feats, new_feats)

                if deltas:
                    intent_delta_episode = derive_delta_intents(deltas)

                    subject_str = cur.get("subject","")
                    ts_epoch_utc = ensure_utc_epoch(cur.get("timestamp", 0))
                    ts_iso_utc   = epoch_to_iso_utc(ts_epoch_utc)

                    # Step-1 labels/scope from *new* snapshot
                    new_labels = _get_labels(new_feats)
                    new_scope  = _get_scope(new_feats)

                    file_kind = new_labels.get("file_kind", "")
                    ci_emu_change = new_labels.get("ci_emulator_step_change", None)
                    gmd_block_change = new_labels.get("gradle_gmd_block_change", None)
                    emulator_keys = _scope_keys(new_scope, "emulator_related")
                    context_keys  = _scope_keys(new_scope, "context_only")
                    is_emulator_file = _is_emulator_relevant_file(new_labels, new_scope)

                    for d in deltas:
                        field = d["field"]
                        key = (path, field)
                        repeat_counter[key] = repeat_counter.get(key, 0) + 1
                        repeat_index = repeat_counter[key]
                        repeat_label = "first" if repeat_index == 1 else "repeat"

                        # Primary (WHAT) and Category
                        primary_label = FIELD_TO_PRIMARY.get(field, "other_change")
                        primary_category = CATEGORY_NAME.get(
                            PRIMARY_TO_CATEGORY.get(primary_label, CAT_CONTEXT),
                            PRIMARY_TO_CATEGORY.get(primary_label, CAT_CONTEXT)
                        )

                        # Per-source intents
                        if field in DELTA_MEANINGFUL_FIELDS:
                            intents_delta_list = derive_delta_intents([d])
                        else:
                            intents_delta_list = []

                        intent_path_generic = detect_intent_path_generic(path)
                        intent_pfa_list, intent_pfa_reason = detect_intent_path_field_aware(path, field, subject_str, d)
                        intent_subject_all, aux_subject, subj_reason = detect_subject_intents(subject_str)

                        intents_delta   = list(sorted(set(intents_delta_list)))
                        intents_subject = list(sorted(set(intent_subject_all)))
                        intents_path    = list(sorted(set(intent_path_generic)))
                        intents_pfa     = list(sorted(set(intent_pfa_list)))

                        # Arbitration
                        source_to_pool = [
                            ("Delta",   intents_delta,   "delta-derived"),
                            ("PathFA",  intents_pfa,     intent_pfa_reason or "path field-aware"),
                            ("Subject", intents_subject, subj_reason or "subject regex"),
                            ("Path",    intents_path,    "path generic"),
                        ]
                        chosen_source = None
                        chosen_raws: List[str] = []
                        chosen_reason = None
                        for s, pool, why in source_to_pool:
                            if pool:
                                chosen_source = s
                                chosen_raws = pool
                                chosen_reason = why
                                break

                        secondary_label = ",".join(chosen_raws) if chosen_raws else ""
                        sub_cands = to_subbuckets(chosen_raws)
                        driver = pick_by_priority(sub_cands) if sub_cands else ""
                        headline = SUB_TO_HEADLINE.get(driver, "") if driver else ""

                        # change_op
                        change_op = classify_change_op(d.get("old_value",""), d.get("new_value",""), d.get("change_type","modified"))

                        # --- Scope handling (effective scope for conditional runtime fields)
                        old_scope = _get_scope(old_feats)
                        old_field_scope = (old_scope.get(field) or "").strip().lower()
                        new_field_scope = (new_scope.get(field) or "").strip().lower()
                        effective_field_scope = effective_scope_for_conditional(field, old_field_scope, new_field_scope)

                        # Spot + intent label from effective scope
                        spot_tag = _spot_tag(file_kind, effective_field_scope)
                        intent_label = _intent_label(primary_label, primary_category, file_kind, field, effective_field_scope)

                        # Include flag for emulator evolution study
                        include_emu_study = (
                            effective_field_scope == "emulator_related"
                            or primary_category in {CAT_COVERAGE, CAT_INVOC, CAT_ORCH}
                        )

                        out_rows.append({
                            # --- Metadata
                            "repo": repo,
                            "sha": cur.get("sha"),
                            "prev_sha": prev.get("sha"),
                            "timestamp_epoch_utc": ts_epoch_utc,
                            "timestamp_utc": ts_iso_utc,
                            "path": path,
                            "subject": subject_str,

                            # --- Primary (WHAT)
                            "Field": field,
                            "Primary_Label": primary_label,
                            "Primary_Label_Category": primary_category,

                            # --- Diff details
                            "old_value": d.get("old_value",""),
                            "new_value": d.get("new_value",""),
                            "change_type": d.get("change_type","modified"),
                            "change_op": change_op,
                            "magnitude": d.get("magnitude", None),
                            "added_items": d.get("added_items",""),
                            "removed_items": d.get("removed_items",""),
                            "repeat_index": repeat_index,
                            "repeat_label": repeat_label,

                            # --- Per-source intents
                            "intent_delta": ",".join(intents_delta),
                            "intent_pfa": ",".join(intents_pfa),
                            "intent_subject": ",".join(intents_subject),
                            "intent_path": ",".join(intents_path),
                            "intent_delta_episode": ",".join(sorted(set(intent_delta_episode))) if intent_delta_episode else "",

                            # --- Secondary (chosen source only)
                            "Secondary_Label": secondary_label,

                            # --- Why (7 + 4)
                            "Driver": driver,
                            "intent_headline": headline,
                            "Driver_Source": chosen_source or "",
                            "Driver_reason": chosen_reason or "",

                            # --- Aux
                            "Aux_Tags": ",".join(aux_subject) if aux_subject else "",
                            "Intent_Path": "; ".join(intent_pfa_list) if intent_pfa_list else None,
                            "Intent_Path_Reason": intent_pfa_reason,
                            "Intent_Subject_Reason": subj_reason,

                            # --- Drivers (legacy compat)
                            "Driver_Label_Selected": driver,

                            # --- Step-1 labels/scope carried through
                            "File_Kind": file_kind,                                 # ci_yaml|gradle|script|other
                            "CI_Emulator_Step_Change": ci_emu_change,               # True/False/None
                            "Gradle_GMD_Block_Change": gmd_block_change,            # True/False/None
                            "Field_Scope": new_field_scope,                          # raw new scope from Step-1
                            "Emulator_Scope_Keys": ",".join(emulator_keys),
                            "Context_Scope_Keys": ",".join(context_keys),
                            "Is_Emulator_Relevant_File": bool(is_emulator_file),

                            # --- Effective scope + intent/spot
                            "Old_Field_Scope": old_field_scope,
                            "New_Field_Scope": new_field_scope,
                            "Effective_Field_Scope": effective_field_scope,
                            "Spot_Tag": spot_tag,                                   # yaml_scoped|yaml_unscoped|gradle_gmd|gradle_global|None
                            "Intent_Label": intent_label,                           # emulator_intent|context_only

                            # --- Study filter
                            "Include_Emulator_Study": bool(include_emu_study),
                        })
                prev = cur

        if not out_rows:
            print(f"[skip] {repo}: no field-level deltas found")
            continue

        dst = OUT_DIR / f"{repo}.jsonl"
        with dst.open("w", encoding="utf-8") as f:
            for r in out_rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"[ok] {repo}: {len(out_rows)} enriched rows -> {dst}")

    print("Done.")


[info] snapshots dir: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots
[info] output dir   : C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V4.7
[info] Loaded snapshots for 282 repos
[ok] 4eRTuk__audioview: 10 enriched rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V4.7\4eRTuk__audioview.jsonl
[ok] a-mabe__OpenHIIT: 16 enriched rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V4.7\a-mabe__OpenHIIT.jsonl
[ok] a914-gowtham__compose-ratingbar: 18 enriched rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V4.7\a914-gowtham__compose-ratingbar.jsonl
[ok] AAkira__ExpandableLayout: 5 enriched rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V4.7\AAkira__ExpandableLayout.jsonl
[ok] abdelaziz-mahdy__pytorch_lite: 24 enriched rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V4.7\abdelaziz-mahdy__pytorch_lite.json

In [5]:
# combine_v47.py
# RQ2 — Step 3 (Combine V4.7): Snapshots + Enriched Episodes
# - Keeps Step-2 V4.7 schema only (no legacy aliasing)
# - Column order: Metadata → Other → Primary → Secondary → Driver
# - Cleans list-like cells to CSV-friendly strings
# - Includes Step-1 carry-over + effective scope fields from Step-2
# - NO Parquet output (per request)

from __future__ import annotations
import json, csv, ast, math
from pathlib import Path
from typing import List, Any

# -----------------------------
# Config
# -----------------------------
WORK_ROOT         = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
SNAPSHOT_DIR      = WORK_ROOT / "snapshots"
CCE_ENRICHED_DIR  = WORK_ROOT / "cce_enriched_V4.7"
COMBINE_DIR       = WORK_ROOT / "combined_V4.7"
COMBINE_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Helpers
# -----------------------------
def read_all_jsonl(folder: Path) -> List[dict]:
    out: List[dict] = []
    if not folder.exists():
        return out
    for p in folder.glob("*.jsonl"):
        with p.open(encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    d = json.loads(line)
                    d["_source_file"] = p.name
                    out.append(d)
    return out

def to_json(x: Any) -> str:
    try:
        return json.dumps(x, ensure_ascii=False, sort_keys=True)
    except Exception:
        return "" if x is None else str(x)

def _is_na_like(x: Any) -> bool:
    if x is None:
        return True
    if isinstance(x, float):
        try:
            return math.isnan(x)
        except Exception:
            return False
    if isinstance(x, str):
        s = x.strip().lower()
        return s in {"", "null", "none", "nan", "na"}
    return False

def conservative_change_op(old_v: Any, new_v: Any) -> str:
    o_empty = _is_na_like(old_v)
    n_empty = _is_na_like(new_v)
    if o_empty and n_empty:
        return "no_change"
    if o_empty and not n_empty:
        return "add"
    if not o_empty and n_empty:
        return "remove"
    return "value_edit" if (str(old_v) != str(new_v)) else "no_change"

def to_list_like(x: Any) -> list:
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, (set, tuple)):
        return list(x)
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        # JSON- or Python-like list
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                v = json.loads(s)
                if isinstance(v, list):
                    return v
            except Exception:
                pass
            try:
                v = ast.literal_eval(s)
                if isinstance(v, (list, tuple, set)):
                    return list(v)
            except Exception:
                pass
        # Delimited
        if "," in s:
            return [t.strip() for t in s.split(",") if t.strip()]
        if ";" in s:
            return [t.strip() for t in s.split(";") if t.strip()]
        # Single token
        return [s]
    # Fallback
    return [x]

def clean_cell_no_brackets(v: Any) -> str:
    """Make cells CSV-friendly: lists→'a,b,c', dicts→JSON, strip brackets for stringified lists."""
    if v is None:
        return ""
    if isinstance(v, bool):
        return "True" if v else "False"
    if isinstance(v, dict):
        try:
            return json.dumps(v, ensure_ascii=False, sort_keys=True)
        except Exception:
            return str(v)
    if isinstance(v, (list, tuple, set)) or (isinstance(v, str) and v.strip()[:1] in "[(" and v.strip()[-1:] in "])"):
        items = [str(t).strip() for t in to_list_like(v) if str(t).strip()]
        items = sorted(set(items))
        return ",".join(items)
    s = str(v).strip()
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        items = [str(t).strip() for t in to_list_like(s) if str(t).strip()]
        items = sorted(set(items))
        return ",".join(items)
    return str(v)

# -----------------------------
# Load inputs
# -----------------------------
snapshots = read_all_jsonl(SNAPSHOT_DIR)
episodes  = read_all_jsonl(CCE_ENRICHED_DIR)
print(f"Loaded {len(snapshots)} snapshots; {len(episodes)} enriched episode rows.")

# -----------------------------
# Write CSV (snapshots)
# -----------------------------
snap_csv = COMBINE_DIR / "snapshots_combined.csv"
if snapshots:
    snaps_flat = []
    for r in snapshots:
        feats = r.get("features", {})
        rr = {**r, "features_json": to_json(feats)}
        rr.pop("features", None)
        snaps_flat.append(rr)

    keys = sorted(set().union(*[set(x.keys()) for x in snaps_flat]))
    with snap_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in snaps_flat:
            r_clean = {k: clean_cell_no_brackets(v) for k, v in r.items()}
            w.writerow(r_clean)
    print(f"[ok] {snap_csv}")
else:
    print("[warn] No snapshots found.")

# -----------------------------
# Write CSV (episodes enriched) — align to Step-2 V4.7
# -----------------------------
cce_csv = COMBINE_DIR / "episodes_enriched_combined.csv"
if episodes:
    # Only current schema (V4.7). No legacy fields.
    # Groups → Metadata / Other / Primary / Secondary / Driver
    META_COLS = [
        "repo", "_source_file", "sha", "prev_sha",
        "timestamp_epoch_utc", "timestamp_utc",
        "path", "subject",
    ]

    # Other includes: diff/audit, Step-1 carryover, effective scope & study flags
    DIFF_COLS = [
        "old_value", "new_value", "change_type", "change_op", "magnitude",
        "added_items", "removed_items", "repeat_index", "repeat_label",
        "Aux_Tags",
        "intent_delta_episode",     # episode-union of delta intents (from Step-2)
    ]

    STEP1_CARRY_COLS = [
        # Step-1 labels/scope carried through
        "File_Kind",                      # ci_yaml|gradle|script|other
        "CI_Emulator_Step_Change",        # True/False/None
        "Gradle_GMD_Block_Change",        # True/False/None
        "Field_Scope",                    # raw NEW scope from Step-1 for this field
        "Emulator_Scope_Keys",            # csv
        "Context_Scope_Keys",             # csv
        "Is_Emulator_Relevant_File",      # bool
    ]

    EFFECTIVE_SCOPE_COLS = [
        "Old_Field_Scope",
        "New_Field_Scope",
        "Effective_Field_Scope",
        "Spot_Tag",                       # yaml_scoped|yaml_unscoped|gradle_gmd|gradle_global|None
        "Intent_Label",                   # emulator_intent|context_only (tables A–E)
        "Include_Emulator_Study",         # bool filter flag
    ]

    OTHER_COLS = DIFF_COLS + STEP1_CARRY_COLS + EFFECTIVE_SCOPE_COLS

    PRIMARY_COLS = [
        "Field",
        "Primary_Label",
        "Primary_Label_Category",
    ]

    SECONDARY_COLS = [
        # Per-source intents & chosen
        "intent_delta", "intent_pfa", "intent_subject", "intent_path",
        "Secondary_Label",
        # Reasons/aux from Step-2 (preserved when present)
        "Intent_Path", "Intent_Path_Reason", "Intent_Subject_Reason",
    ]

    DRIVER_COLS = [
        "Driver",
        "intent_headline",
        "Driver_Source",
        "Driver_reason",
        "Driver_Label_Selected",   # keep for compatibility/debug
    ]

    ORDERED_COLS = META_COLS + OTHER_COLS + PRIMARY_COLS + SECONDARY_COLS + DRIVER_COLS

    # Build rows (only fields from current schema; unknown keys ignored)
    rows_out: List[dict] = []
    for r in episodes:
        rr = {}
        for k in ORDERED_COLS:
            if k in r:
                rr[k] = r.get(k, "")
        # conservative fill for change_op if missing
        if not rr.get("change_op"):
            rr["change_op"] = conservative_change_op(r.get("old_value"), r.get("new_value"))
        # clean cells
        rr = {k: clean_cell_no_brackets(v) for k, v in rr.items()}
        rows_out.append(rr)

    # Header: keep ORDERED_COLS but only include columns that appear at least once
    present_cols = [c for c in ORDERED_COLS if any((c in r and r[c] != "") for r in rows_out)]
    # Ensure core metadata always present in header
    for c in META_COLS:
        if c not in present_cols:
            present_cols.insert(0 if c == "repo" else len(present_cols), c)

    with cce_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=present_cols)
        w.writeheader()
        for r in rows_out:
            w.writerow({k: r.get(k, "") for k in present_cols})

    print(f"[ok] {cce_csv}")
else:
    print("[warn] No enriched episodes found.")


Loaded 185593 snapshots; 11586 enriched episode rows.
[ok] C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined_V4.7\snapshots_combined.csv
[ok] C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined_V4.7\episodes_enriched_combined.csv
